# Logistics Operations — Data Cleaning
**Project:** Logistics Capstone  
**Notebook:** 02 — Clean & Prepare Data  
**Tools:** pandas  

---
**Goal:** Fix the 4 null issues identified in notebook 01, confirm data types are correct, and export clean CSVs ready for analysis and Tableau.

## 1. Import libraries and reload data

In [11]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

DATA_PATH    = '../data/'
EXPORT_PATH  = '../exports/'

drivers          = pd.read_csv(DATA_PATH + 'drivers.csv')
trucks           = pd.read_csv(DATA_PATH + 'trucks.csv')
trailers         = pd.read_csv(DATA_PATH + 'trailers.csv')
customers        = pd.read_csv(DATA_PATH + 'customers.csv')
facilities       = pd.read_csv(DATA_PATH + 'facilities.csv')
routes           = pd.read_csv(DATA_PATH + 'routes.csv')
loads            = pd.read_csv(DATA_PATH + 'loads.csv')
trips            = pd.read_csv(DATA_PATH + 'trips.csv')
fuel_purchases   = pd.read_csv(DATA_PATH + 'fuel_purchases.csv')
maintenance      = pd.read_csv(DATA_PATH + 'maintenance_records.csv')
delivery_events  = pd.read_csv(DATA_PATH + 'delivery_events.csv')
safety_incidents = pd.read_csv(DATA_PATH + 'safety_incidents.csv')
driver_metrics   = pd.read_csv(DATA_PATH + 'driver_monthly_metrics.csv')
truck_metrics    = pd.read_csv(DATA_PATH + 'truck_utilization_metrics.csv')

print('All 14 tables loaded.')

All 14 tables loaded.


## 2. Fix nulls

From notebook 01 we identified 4 issues to fix:
1. `drivers.termination_date` — fill with `'Active'`
2. `trips` — drop rows where driver_id / truck_id / trailer_id is null
3. `fuel_purchases` — drop rows where truck_id or driver_id is null
4. `safety_incidents` — drop the 2 rows with nulls

### 2a. DRIVERS — fill termination_date

In [2]:
print('Before:', drivers['termination_date'].isnull().sum(), 'nulls')

drivers['termination_date'] = drivers['termination_date'].fillna('Active')

print('After: ', drivers['termination_date'].isnull().sum(), 'nulls')
print()
print('Value counts:')
print(drivers['termination_date'].value_counts().head(10))

Before: 124 nulls
After:  0 nulls

Value counts:
termination_date
Active        124
2013-01-08      1
2019-02-27      1
2014-12-19      1
2014-02-03      1
2021-02-17      1
2015-11-15      1
2016-05-19      1
2016-11-21      1
2018-11-18      1
Name: count, dtype: int64


### 2b. TRIPS — drop rows with null driver_id, truck_id, or trailer_id

In [3]:
print(f'Before: {trips.shape[0]:,} rows')

trips_clean = trips.dropna(subset=['driver_id', 'truck_id', 'trailer_id']).copy()

dropped = trips.shape[0] - trips_clean.shape[0]
print(f'After:  {trips_clean.shape[0]:,} rows  ({dropped:,} rows dropped)')
print(f'Nulls remaining: {trips_clean.isnull().sum().sum()}')

Before: 85,410 rows
After:  80,458 rows  (4,952 rows dropped)
Nulls remaining: 0


### 2c. FUEL_PURCHASES — drop rows with null truck_id or driver_id

In [4]:
print(f'Before: {fuel_purchases.shape[0]:,} rows')

fuel_clean = fuel_purchases.dropna(subset=['truck_id', 'driver_id']).copy()

dropped = fuel_purchases.shape[0] - fuel_clean.shape[0]
print(f'After:  {fuel_clean.shape[0]:,} rows  ({dropped:,} rows dropped)')
print(f'Nulls remaining: {fuel_clean.isnull().sum().sum()}')

Before: 196,442 rows
After:  188,668 rows  (7,774 rows dropped)
Nulls remaining: 0


### 2d. SAFETY_INCIDENTS — drop the 2 rows with nulls

In [5]:
print(f'Before: {safety_incidents.shape[0]:,} rows')

safety_clean = safety_incidents.dropna(subset=['truck_id', 'driver_id']).copy()

dropped = safety_incidents.shape[0] - safety_clean.shape[0]
print(f'After:  {safety_clean.shape[0]:,} rows  ({dropped:,} rows dropped)')
print(f'Nulls remaining: {safety_clean.isnull().sum().sum()}')

Before: 170 rows
After:  168 rows  (2 rows dropped)
Nulls remaining: 0


## 3. Check and fix data types

Date columns are often read as strings by pandas — we need to convert them  
so we can do time-based analysis (monthly trends, on-time calculations etc.)

### 3a. Check current data types on key tables

In [6]:
print('--- TRIPS ---')
print(trips_clean.dtypes)
print()
print('--- LOADS ---')
print(loads.dtypes)
print()
print('--- DELIVERY_EVENTS ---')
print(delivery_events.dtypes)

--- TRIPS ---
trip_id                   object
load_id                   object
driver_id                 object
truck_id                  object
trailer_id                object
dispatch_date             object
actual_distance_miles      int64
actual_duration_hours    float64
fuel_gallons_used        float64
average_mpg              float64
idle_time_hours          float64
trip_status               object
dtype: object

--- LOADS ---
load_id                 object
customer_id             object
route_id                object
load_date               object
load_type               object
weight_lbs               int64
pieces                   int64
revenue                float64
fuel_surcharge         float64
accessorial_charges      int64
load_status             object
booking_type            object
dtype: object

--- DELIVERY_EVENTS ---
event_id              object
load_id               object
trip_id               object
event_type            object
facility_id           object
sched

### 3b. Convert date columns

`pd.to_datetime()` converts string dates to proper datetime objects.  
`errors='coerce'` turns any unparseable values into NaT (null) instead of crashing.

Update the column names below if your actual column names differ — check the dtypes output above.

In [7]:
# Trips — convert date/time columns (exclude idle_time_hours)
for col in trips_clean.columns:
    if ('date' in col.lower() or 'time' in col.lower()) and col != 'idle_time_hours':
        trips_clean[col] = pd.to_datetime(trips_clean[col], errors='coerce')
        print(f'trips_clean.{col} → {trips_clean[col].dtype}')

# Loads
for col in loads.columns:
    if 'date' in col.lower() or 'time' in col.lower():
        loads[col] = pd.to_datetime(loads[col], errors='coerce')
        print(f'loads.{col} → {loads[col].dtype}')

# Delivery events (exclude on_time_flag)
for col in delivery_events.columns:
    if ('date' in col.lower() or 'time' in col.lower()) and col != 'on_time_flag':
        delivery_events[col] = pd.to_datetime(delivery_events[col], errors='coerce')
        print(f'delivery_events.{col} → {delivery_events[col].dtype}')

# Drivers termination date
# We skip this one — it now contains 'Active' strings mixed with dates

print('\nDate conversions done.')

trips_clean.dispatch_date → datetime64[ns]
loads.load_date → datetime64[ns]
delivery_events.scheduled_datetime → datetime64[ns]
delivery_events.actual_datetime → datetime64[ns]

Date conversions done.


## 4. Final null check — confirm everything is clean

In [8]:
clean_tables = {
    'DRIVERS':                  drivers,
    'TRUCKS':                   trucks,
    'TRAILERS':                 trailers,
    'CUSTOMERS':                customers,
    'FACILITIES':               facilities,
    'ROUTES':                   routes,
    'LOADS':                    loads,
    'TRIPS (clean)':            trips_clean,
    'FUEL_PURCHASES (clean)':   fuel_clean,
    'MAINTENANCE_RECORDS':      maintenance,
    'DELIVERY_EVENTS':          delivery_events,
    'SAFETY_INCIDENTS (clean)': safety_clean,
    'DRIVER_MONTHLY_METRICS':   driver_metrics,
    'TRUCK_UTILIZATION_METRICS':truck_metrics,
}

print(f'{"Table":<35} {"Rows":>8} {"Nulls":>8}')
print('-' * 54)
for name, df in clean_tables.items():
    nulls = df.isnull().sum().sum()
    flag  = '  <- still has nulls' if nulls > 0 else ''
    print(f'{name:<35} {df.shape[0]:>8,} {nulls:>8,}{flag}')

Table                                   Rows    Nulls
------------------------------------------------------
DRIVERS                                  150        0
TRUCKS                                   120        0
TRAILERS                                 180        0
CUSTOMERS                                200        0
FACILITIES                                50        0
ROUTES                                    58        0
LOADS                                 85,410        0
TRIPS (clean)                         80,458        0
FUEL_PURCHASES (clean)               188,668        0
MAINTENANCE_RECORDS                    2,920        0
DELIVERY_EVENTS                      170,820        0
SAFETY_INCIDENTS (clean)                 168        0
DRIVER_MONTHLY_METRICS                 4,464        0
TRUCK_UTILIZATION_METRICS              3,312        0


## 5. Export clean CSVs for Tableau

We export the cleaned versions of the 3 tables we modified,  
plus the key reference tables that will be joined in Tableau.

`index=False` stops pandas writing a row number column into the CSV.

In [9]:
# Cleaned transactional tables
trips_clean.to_csv(EXPORT_PATH + 'trips_clean.csv',            index=False)
fuel_clean.to_csv(EXPORT_PATH  + 'fuel_purchases_clean.csv',   index=False)
safety_clean.to_csv(EXPORT_PATH + 'safety_incidents_clean.csv',index=False)

# Reference tables (already clean — just copy to exports for Tableau)
drivers.to_csv(EXPORT_PATH         + 'drivers.csv',         index=False)
trucks.to_csv(EXPORT_PATH          + 'trucks.csv',          index=False)
trailers.to_csv(EXPORT_PATH        + 'trailers.csv',        index=False)
customers.to_csv(EXPORT_PATH       + 'customers.csv',       index=False)
facilities.to_csv(EXPORT_PATH      + 'facilities.csv',      index=False)
routes.to_csv(EXPORT_PATH          + 'routes.csv',          index=False)
loads.to_csv(EXPORT_PATH           + 'loads.csv',           index=False)
maintenance.to_csv(EXPORT_PATH     + 'maintenance_records.csv', index=False)
delivery_events.to_csv(EXPORT_PATH + 'delivery_events.csv', index=False)
driver_metrics.to_csv(EXPORT_PATH  + 'driver_monthly_metrics.csv',    index=False)
truck_metrics.to_csv(EXPORT_PATH   + 'truck_utilization_metrics.csv', index=False)

print('All clean CSVs exported to exports/ folder.')

All clean CSVs exported to exports/ folder.


## 6. Confirm exports exist

In [10]:
import os

files = sorted(os.listdir(EXPORT_PATH))
print(f'{len(files)} files in exports/:')
for f in files:
    size = os.path.getsize(EXPORT_PATH + f) / 1024
    print(f'  {f:<45} {size:>8.1f} KB')

22 files in exports/:
  analysis_driver_performance.csv                    5.8 KB
  analysis_fleet_health.csv                          5.0 KB
  analysis_route_profitability.csv                   5.3 KB
  chart_driver_mpg.png                              42.9 KB
  chart_driver_ontime.png                           99.2 KB
  chart_fleet_health.png                            95.8 KB
  chart_route_profitability.png                     57.1 KB
  customers.csv                                     14.8 KB
  delivery_events.csv                            21718.8 KB
  driver_monthly_metrics.csv                       259.1 KB
  drivers.csv                                       13.4 KB
  facilities.csv                                     4.2 KB
  fuel_purchases_clean.csv                       19171.4 KB
  loads.csv                                       8187.5 KB
  maintenance_records.csv                          307.4 KB
  placeholder.md                                     0.0 KB
  routes.csv      

---
**Next notebook:** `03_analysis.ipynb` — merge tables and answer the 3 analysis questions:  
1. Driver performance (on-time rate, MPG, revenue per mile)  
2. Route profitability (revenue vs fuel cost by lane)  
3. Fleet health (utilization, maintenance cost per mile)